one-hot encoding + Apriori, generate frequent_itemsets
exports frequent_itemsets

In [3]:
#import library for start the model creation
import pandas as pd
import ast
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori

In [4]:
# load prior basket from outputs
prior_baskets = pd.read_csv("../outputs/prior_baskets.csv")
prior_baskets["basket"] = prior_baskets["basket"].apply(ast.literal_eval) # Convert from a string to a real Python list

In [5]:
# transform all into a list of transactions with 1 transaction is 1 list of product_id
transactions = prior_baskets["basket"].tolist()
print("Exemple transaction:", transactions[0][:10])

Exemple transaction: [1819, 9327, 17794, 28985, 33120]


In [6]:
# One hot encoding (mlxtend) transactions to have True or False matrix where rows=baskets and columns=products
te = TransactionEncoder()
te_ary = te.fit(transactions).transform(transactions)
onehot = pd.DataFrame(te_ary, columns=te.columns_)

In [7]:
#verify the size and print the one-hot matrix
print("onehot shape:", onehot.shape)
display(onehot.head())

onehot shape: (3115534, 3000)


,1,10,25,34,45,49,79,95,116,117,...,49519,49520,49533,49583,49585,49605,49610,49621,49628,49683
0,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
1,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
2,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
3,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
4,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False


In [8]:
# Apriori
#minimum support controling how frequent itemset must be to be kept
MIN_SUPPORT = 0.002  # (itemset must appear in atleast 0.2% of basket to be kept)

# runing Apriori on the one-hot matrix to have frequent itemsets and their support.
frequent_itemsets = apriori(
    onehot,
    min_support=MIN_SUPPORT,
    use_colnames=True,
    low_memory=True,
    max_len=2
)
#adding an extra column indicating the number of products
frequent_itemsets["length"] = frequent_itemsets["itemsets"].apply(len)

print("nb itemsets:", len(frequent_itemsets))
display(frequent_itemsets.sort_values(["support", "length"], ascending=[False, True]).head(10))

print("nb itemsets:", len(frequent_itemsets))
display(frequent_itemsets.sort_values(["support", "length"], ascending=[False, True]).head(10))


nb itemsets: 1467


,support,itemsets,length
441,0.151680,(24852),1
224,0.121793,(13176),1
370,0.084956,(21137),1
389,0.077650,(21903),1
834,0.068555,(47209),1
843,0.056753,(47766),1
838,0.048999,(47626),1
288,0.045883,(16797),1
462,0.045137,(26209),1
491,0.044264,(27845),1


nb itemsets: 1467


,support,itemsets,length
441,0.151680,(24852),1
224,0.121793,(13176),1
370,0.084956,(21137),1
389,0.077650,(21903),1
834,0.068555,(47209),1
843,0.056753,(47766),1
838,0.048999,(47626),1
288,0.045883,(16797),1
462,0.045137,(26209),1
491,0.044264,(27845),1


# Changements made after runing
At first exemple, there was an error saying : "nable to allocate 2.16 TiB for an array with shape (381501, 2, 3115534)"
to correct the low_memory=True and max_len=2 was add to frequent_itemsets

In [9]:
# save the frequent itemsets into csv
frequent_itemsets.to_csv("../outputs/frequent_itemsets.csv", index=False)

In [10]:
# verify that only 2 were kept
frequent_itemsets["length"].value_counts()

length
1    874
2    593
Name: count, dtype: int64